# HW4 — Aggregator inflation on the $C_6$ vs $2K_3$ blind spot

**Goal.** Show that no permutation-invariant node-local aggregator (sum, mean, max) separates $C_6$ from $2K_3$ under constant or degree features. Triangle counts globally distinguish them but remain constant *within* each graph, explaining why genuine 1-WL needs multisets of *colours*, not scalars.

## Setup

Resolve `onboarding/projects/` on `sys.path`, attach the reflection log, and import the canonical references (used only for spot-checks; the student version derives its own implementations).

In [ ]:
import os, sys
from pathlib import Path
_here = Path(os.getcwd()).resolve()
for _p in [_here, *_here.parents]:
    if (_p / 'shared' / '__init__.py').exists() and _p.name == 'projects':
        _PROJECTS = _p
        break
else:
    raise RuntimeError('could not locate onboarding/projects/')
_REPO = _PROJECTS.parent.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless for CI
import matplotlib.pyplot as plt

from onboarding.projects import reflect
reflect.start(notebook='hw4', store=_PROJECTS / '.reflection.jsonl')
print(f'[setup] projects root: {_PROJECTS}')


In [ ]:
def edges_C6():
    pairs = [(i,(i+1)%6) for i in range(6)]
    e = []
    for u,v in pairs:
        e += [(u,v),(v,u)]
    return np.array(e).T

def edges_2K3():
    pairs = [(0,1),(1,2),(0,2),(3,4),(4,5),(3,5)]
    e = []
    for u,v in pairs:
        e += [(u,v),(v,u)]
    return np.array(e).T


## Q1 — Three aggregator partitions.

`sum/mean/max_partition(edge_index, n, x)` groups nodes by `(x_u, σ_u)` where $σ_u$ aggregates over $\{x_v : v \sim u\}$.

In [ ]:
def _neighbours(edge_index, n):
    nbrs = [[] for _ in range(n)]
    for u, v in zip(edge_index[0], edge_index[1]):
        nbrs[int(u)].append(int(v))
    return nbrs

def _partition_by_key(keys):
    table = {}
    for u, k in enumerate(keys):
        table.setdefault(k, []).append(u)
    return [np.array(v) for v in table.values()]

def sum_partition(edge_index, n, x):
    nbrs = _neighbours(edge_index, n)
    keys = [(float(x[u]), float(sum(x[v] for v in nbrs[u]))) for u in range(n)]
    return _partition_by_key(keys)

def mean_partition(edge_index, n, x):
    nbrs = _neighbours(edge_index, n)
    keys = [(float(x[u]), float(np.mean([x[v] for v in nbrs[u]])) if nbrs[u] else 0.0) for u in range(n)]
    return _partition_by_key(keys)

def max_partition(edge_index, n, x):
    nbrs = _neighbours(edge_index, n)
    keys = [(float(x[u]), float(max([x[v] for v in nbrs[u]])) if nbrs[u] else 0.0) for u in range(n)]
    return _partition_by_key(keys)


**Distinguish — partition vs label.** The aggregator groups *nodes*; it does not assign class labels. Partition cells coarsen as the key set shrinks.

In [ ]:
# K2 with x=[1,1]: each aggregator → one cell.
ei_k2 = np.array([[0,1],[1,0]])
for name, fn in [('sum',sum_partition),('mean',mean_partition),('max',max_partition)]:
    p = fn(ei_k2, 2, np.array([1.0,1.0]))
    print(f'  K2 / {name}: {len(p)} cell(s)')


**Gate Q1.** All three aggregators yield 1 cell on $K_2$ with $x=(1,1)$.

In [ ]:
for name, fn in [('sum',sum_partition),('mean',mean_partition),('max',max_partition)]:
    p = fn(ei_k2, 2, np.array([1.0,1.0]))
    assert len(p) == 1, f'Q1: {name} should collapse K2 to 1 cell'
print('[GATE OK] Q1: sum/mean/max collapse K2 with constant features to 1 cell')


In [ ]:
reflect.log('Q4.Q1_aggregators', 'sum/mean/max partitions implemented; collapse on K2/constant', 'HIGH')


## Q2 — Constant features collapse both $C_6$ and $2K_3$ under every aggregator.

In [ ]:
x_const = np.ones(6)
ei_c6, ei_k3 = edges_C6(), edges_2K3()
results = {}
for name, fn in [('sum',sum_partition),('mean',mean_partition),('max',max_partition)]:
    for g, ei in [('C6', ei_c6), ('2K3', ei_k3)]:
        p = fn(ei, 6, x_const)
        results[(name,g)] = (len(p), tuple(sorted(len(c) for c in p)))
        print(f'  {g} / {name}: {len(p)} cell(s), sizes {results[(name,g)][1]}')


**Gate Q2.** Every (aggregator × graph) pair returns exactly 1 cell of size 6.

In [ ]:
for k, (ncells, sizes) in results.items():
    assert ncells == 1 and sizes == (6,), f'Q2: {k} should be a single 6-cell, got {ncells},{sizes}'
print('[GATE OK] Q2: constant features → 1×6 on both C6 and 2K3 under sum/mean/max')


In [ ]:
reflect.log('Q4.Q2_constant_collapse', 'all 6 (agg×graph) cases collapse to 1×6', 'HIGH')


## Q3 — Degree feature. Both graphs are 2-regular, so feature is constant ⇒ still blind.

In [ ]:
def degree(edge_index, n):
    deg = np.zeros(n)
    for u in edge_index[0]:
        deg[int(u)] += 1
    return deg

deg_c6, deg_k3 = degree(ei_c6, 6), degree(ei_k3, 6)
print(f'deg(C6)={deg_c6}, deg(2K3)={deg_k3}')


**Gate Q3.** Both degree vectors equal $(2,2,2,2,2,2)$, so the result is identical to Q2.

In [ ]:
assert np.all(deg_c6 == 2) and np.all(deg_k3 == 2), f'Q3: not 2-regular: {deg_c6}, {deg_k3}'
# As consequence, sum/mean/max with degree feature also collapse to 1×6.
for name, fn in [('sum',sum_partition),('mean',mean_partition),('max',max_partition)]:
    for g, ei, deg in [('C6', ei_c6, deg_c6), ('2K3', ei_k3, deg_k3)]:
        p = fn(ei, 6, deg)
        assert len(p) == 1 and len(p[0]) == 6, f'Q3: {g}/{name} did not collapse with degree feature'
print('[GATE OK] Q3: degree feature is constant on both → identical aggregator collapse')


In [ ]:
reflect.log('Q4.Q3_degree_blind', 'degree feature equal to constant on 2-regular graphs', 'HIGH')


## Q4 — Triangle count separates the two graphs globally but is constant within each.

In [ ]:
def triangle_count(edge_index, n):
    nbrs = [set() for _ in range(n)]
    for u, v in zip(edge_index[0], edge_index[1]):
        nbrs[int(u)].add(int(v))
    out = np.zeros(n, dtype=int)
    for u in range(n):
        for v in nbrs[u]:
            if v <= u: continue
            for w in nbrs[u] & nbrs[v]:
                if w > v:
                    out[u] += 1; out[v] += 1; out[w] += 1
    return out


**Distinguish.** Triangle count is a *global* invariant that distinguishes graphs but is constant within each graph here → still no within-graph partition refinement.

In [ ]:
t_c6 = triangle_count(ei_c6, 6)
t_k3 = triangle_count(ei_k3, 6)
print(f'triangles(C6)={t_c6}  sum={t_c6.sum()}')
print(f'triangles(2K3)={t_k3}  sum={t_k3.sum()}')


**Gate Q4.** $C_6$ → all 0; $2K_3$ → all 1; graphs distinguished by feature sum.

In [ ]:
assert np.all(t_c6 == 0), f'Q4: C6 triangle vector {t_c6} ≠ 0'
assert np.all(t_k3 == 1), f'Q4: 2K3 triangle vector {t_k3} ≠ 1'
assert t_c6.sum() != t_k3.sum(), 'Q4: triangle feature did not separate the two graphs'
print('[GATE OK] Q4: triangles(C6)=0 everywhere, triangles(2K3)=1 everywhere')


In [ ]:
reflect.log('Q4.Q4_triangles', 'triangle-count separates C6 from 2K3 globally; constant within each', 'HIGH')


## Q5 — Writeup & calibration.

In [ ]:
reflect.log('Q4.Q5_writeup', 'no permutation-invariant node-local aggregator separates C6 from 2K3', 'HIGH')
print('HW4 reflection log:')
for r in reflect.dump():
    if 'hw4' in r['notebook']:
        print(f"  [{r['level']:>10}] {r['concept']}: {r['claim']}")


**End of HW4.**